# Lab 01 · Simple Linear Regression — by hand, then with scikit-learn

**AI Foundations · Machine Learning · Module 1**  ·  September 2026  ·  about 90 minutes

This lab follows the Module 1 slides step by step. You will fit the *same* straight line three ways — with the formulas, with NumPy, and with scikit-learn — then measure its errors and scale up to a real dataset and a full pipeline.

| Part | What you do | Time |
|---|---|---|
| 0 | Set up the notebook | 5 min |
| 1 | Example 1 by hand: slope and intercept from the formulas | 15 min |
| 2 | The same model in scikit-learn | 10 min |
| 3 | Example 2 — your turn | 10 min |
| 4 | Example 3: noisy data, residuals and error metrics | 20 min |
| 5 | Real data: the seven steps and a pipeline | 20 min |
| 6 | Assessment tasks 1–3 | 10 min + homework |

**How to use this notebook**

1. Open it in **Google Colab** (*File → Upload notebook*) or in **VS Code** with the Jupyter extension. A **CPU runtime is enough** — do not request a GPU.
2. Run cells from top to bottom with **Shift + Enter**.
3. Cells marked **TODO** are yours: replace every `...` with code.
4. Run the **Check** cell straight after each TODO. It tells you whether your answer is right.

## Part 0 · Setup
Every library here is pre-installed on Colab. Run the cell and note the versions — your screen may show newer numbers than the slides, which is fine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.spines.top": False, "axes.spines.right": False})

## Part 1 · Example 1 by hand — slide 20

Simple linear regression fits the line $\hat{y} = m x + b$. Least squares gives the best $m$ and $b$ **directly** — no gradient descent:

$$m = \frac{\sum (x-\bar{x})(y-\bar{y})}{\sum (x-\bar{x})^2} \qquad\qquad b = \bar{y} - m\,\bar{x}$$

Our data: temperatures in °C ($x$) and the matching °F ($y$). The fifth value — 38 °C — is the one we want to predict.

In [ ]:
x = np.array([0, 8, 15, 22], dtype=float)      # °C  (the feature)
y = np.array([32, 46.4, 59, 71.6])             # °F  (the target)

### 1.1 · Build the deviations table
This is the table from slide 21, built with pandas. Read it before moving on: the two deviation columns must each add up to **0** — a free error check.

In [ ]:
x_bar, y_bar = x.mean(), y.mean()

table = pd.DataFrame({"x": x, "y": y, "x - x̄": x - x_bar, "y - ȳ": y - y_bar})
table["(x - x̄)(y - ȳ)"] = table["x - x̄"] * table["y - ȳ"]
table["(x - x̄)²"] = table["x - x̄"] ** 2
table.loc["Σ"] = table.sum()

print(f"x̄ = {x_bar},  ȳ = {y_bar}")
table

### 1.2 · TODO — slope and intercept
Use the **Σ row** of the table and the two formulas. Hint: `table.loc["Σ", "column name"]` reads one cell.

In [ ]:
# TODO: replace each ... using the Σ row and the formulas above
m = ...   # slope
b = ...   # intercept
print("m =", m, "  b =", b)

In [ ]:
# Check
assert np.isclose(m, 1.8) and np.isclose(b, 32), "Not yet — slope = Σ(x−x̄)(y−ȳ) ÷ Σ(x−x̄)², then b = ȳ − m·x̄"
print("PASS: m = 1.8 and b = 32 — the Celsius-to-Fahrenheit formula, learned from four examples")

### 1.3 · TODO — turn the maths into reusable functions
You will reuse `fit_line` and `predict` for every example that follows, so write them once, properly.

In [ ]:
def fit_line(x, y):
    """Least-squares slope and intercept for simple linear regression."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    # TODO: compute the deviations dx and dy, then m and b
    m = ...
    b = ...
    return m, b

def predict(x_new, m, b):
    """Value on the line y = m·x + b."""
    # TODO: return m times x_new plus b
    return ...

m, b = fit_line(x, y)
print("m, b =", m, b, "| 38 °C →", predict(38, m, b), "°F")

In [ ]:
# Check
assert np.allclose(fit_line(x, y), (1.8, 32)), "fit_line should return (1.8, 32) for Example 1"
assert np.isclose(predict(38, 1.8, 32), 100.4), "predict(38, 1.8, 32) should be 100.4"
print("PASS: fit_line and predict work — 38 °C is 100.4 °F")

### 1.4 · The same slope from the “sums” form
Slide 20 also gave a calculator-friendly version: $m = \dfrac{n\sum xy - \sum x \sum y}{n\sum x^2 - (\sum x)^2}$. Both forms are algebraically identical — confirm it:

In [ ]:
n = len(x)
m_sums = (n * (x * y).sum() - x.sum() * y.sum()) / (n * (x ** 2).sum() - x.sum() ** 2)
print("sums form gives m =", m_sums)

## Part 2 · The same model in scikit-learn — slide 40

scikit-learn uses the same three verbs for every model: `fit`, `predict`, `score`.
One rule trips up every beginner: **`X` must be 2-D** — rows × features — even when there is only one feature.

In [ ]:
from sklearn.linear_model import LinearRegression

X = x.reshape(-1, 1)          # 4 rows × 1 feature
print("x shape:", x.shape, "→ X shape:", X.shape)

model = LinearRegression()
model.fit(X, y)                                         # step 4 · training
print("coef_ (m):", model.coef_[0], "| intercept_ (b):", model.intercept_)
print("predict 38 °C:", model.predict([[38]]))         # step 7 · prediction

`1.7999999999999996` is not a mistake — computers store decimals in binary, so tiny rounding appears. Use `np.isclose` rather than `==` to compare floats.

**Break it on purpose.** What happens if you pass the 1-D array?

In [ ]:
try:
    LinearRegression().fit(x, y)        # 1-D x
except ValueError as err:
    print("ValueError:", str(err).splitlines()[0])

### 2.1 · Plot the data, the line and the prediction

In [ ]:
xs = np.linspace(-2, 42, 100).reshape(-1, 1)
plt.scatter(x, y, color="black", zorder=3, label="training data")
plt.plot(xs, model.predict(xs), color="black", label=f"ŷ = {model.coef_[0]:.1f}x + {model.intercept_:.0f}")
plt.scatter([38], model.predict([[38]]), s=90, facecolor="white", edgecolor="black", zorder=4, label="prediction for 38 °C")
plt.xlabel("x  (°C)"); plt.ylabel("y  (°F)"); plt.legend(frameon=False)
plt.show()

## Part 3 · Example 2 — your turn — slides 22–23

$x = -1, 0, 1, 2, 3, 4$ and $y = -3, -1, 1, 3, 5, 7$. Predict $y$ when $x = 10$.

**TODO:** fit the line twice — with your own `fit_line`, and with `LinearRegression` — then predict $x = 10$ with the scikit-learn model.

In [ ]:
x2 = np.array([-1, 0, 1, 2, 3, 4], dtype=float)
y2 = np.array([-3, -1, 1, 3, 5, 7], dtype=float)

# TODO
m2, b2 = ..., ...        # use fit_line
model2 = ...             # LinearRegression fitted on x2 (as 2-D!) and y2
y2_at_10 = ...           # model2's prediction for x = 10, as a single number

print("by hand:", m2, b2, "| x = 10 →", y2_at_10)

In [ ]:
# Check
assert np.allclose((m2, b2), (2, -1)), "fit_line(x2, y2) should give m = 2, b = −1"
assert np.isclose(y2_at_10, 19), "2 × 10 − 1 = 19 — did you reshape x2 and take [0] of the prediction?"
print("PASS: ŷ = 2x − 1, and x = 10 → 19")

**Think.** The training $x$ values run from −1 to 4, but we predicted at 10.
Is that *interpolation* or *extrapolation*? In one sentence, when is it safe?

*Your answer:* …

## Part 4 · Example 3 — noisy data, residuals and errors — slides 25–29

Real data never sits exactly on a line. Five students' study hours ($x$) and exam scores ($y$):

In [ ]:
hours = np.array([1, 2, 3, 4, 5], dtype=float)
score = np.array([50, 58, 57, 66, 69], dtype=float)

m3, b3 = fit_line(hours, score)
y_hat = predict(hours, m3, b3)
e = score - y_hat                                   # residuals: e = y − ŷ

res = pd.DataFrame({"x hours": hours, "y actual": score, "ŷ": y_hat.round(2),
                    "e = y − ŷ": e.round(2), "|e|": np.abs(e).round(2), "e²": (e ** 2).round(2)})
print(f"ŷ = {m3:.1f}x + {b3:.1f}    |    Σe = {e.sum():.1e}  (always ≈ 0 for least squares)")
res

### 4.1 · See the residuals
Each dashed segment is one residual — the **vertical** gap between the point and the line.

In [ ]:
xs = np.linspace(0.5, 5.5, 50)
plt.plot(xs, predict(xs, m3, b3), color="black", label=f"ŷ = {m3:.1f}x + {b3:.1f}")
for xi, yi, hi in zip(hours, score, y_hat):
    plt.plot([xi, xi], [yi, hi], color="black", linestyle="--", linewidth=1)
    plt.annotate(f"e = {yi - hi:+.1f}", (xi + 0.07, (yi + hi) / 2), fontsize=9, color="dimgray")
plt.scatter(hours, score, color="black", zorder=3, label="actual y")
plt.scatter(hours, y_hat, facecolor="white", edgecolor="black", zorder=3, label="predicted ŷ")
plt.xlabel("study hours  (x)"); plt.ylabel("exam score  (y)"); plt.legend(frameon=False)
plt.show()

### 4.2 · TODO — the four error metrics, by hand

$$\text{MAE} = \frac{\sum|e|}{n} \quad \text{MSE} = \frac{\sum e^2}{n} \quad \text{RMSE} = \sqrt{\text{MSE}} \quad R^2 = 1 - \frac{\text{SSE}}{\text{SST}}$$
with $\text{SSE} = \sum e^2$ and $\text{SST} = \sum (y - \bar{y})^2$.

In [ ]:
n = len(score)
# TODO: use e, score and n
SSE = ...
SST = ...

MAE = ...
MSE = ...
RMSE = ...
R2 = ...
print("SSE", SSE, "SST", SST, "| MAE", MAE, "MSE", MSE, "RMSE", RMSE, "R²", R2)

In [ ]:
# Check — compare with scikit-learn's own metric functions
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

library = [mean_absolute_error(score, y_hat), mean_squared_error(score, y_hat),
           root_mean_squared_error(score, y_hat), r2_score(score, y_hat)]
print("scikit-learn:  MAE, MSE, RMSE, R² =", np.round(library, 3))
assert np.allclose([MAE, MSE, RMSE, R2], library), "Your metrics don't match scikit-learn yet — re-check each formula."
print("PASS: your hand-made metrics match scikit-learn: 1.60, 3.68, 1.92, 0.92")

### 4.3 · Is least squares really the *least*?
Let the computer try **every** slope and intercept on a fine grid and compute the SSE of each line. If the formulas are right, the grid's best line lands on the same $m = 4.6$, $b = 46.2$.

In [ ]:
ms = np.linspace(3.0, 6.2, 161)                    # candidate slopes (step 0.02)
bs = np.linspace(40.0, 52.0, 121)                  # candidate intercepts (step 0.1)
M, B = np.meshgrid(ms, bs)
SSE_grid = ((score - (M[..., None] * hours + B[..., None])) ** 2).sum(axis=-1)

i = np.unravel_index(SSE_grid.argmin(), SSE_grid.shape)
print(f"grid search best: m = {M[i]:.2f}, b = {B[i]:.2f}, SSE = {SSE_grid[i]:.2f}")
print(f"formulas:         m = {m3:.2f}, b = {b3:.2f}, SSE = {((score - predict(hours, m3, b3)) ** 2).sum():.2f}")

cs = plt.contour(M, B, SSE_grid, levels=[20, 25, 40, 80, 160, 320], colors="black", linewidths=0.8)
plt.clabel(cs, fontsize=8)
plt.scatter([m3], [b3], color="black", zorder=3)
plt.xlabel("slope m"); plt.ylabel("intercept b"); plt.title("SSE of every line: one bowl, one bottom")
plt.show()

The formulas find the bottom of this bowl in one step. When a model has too many parameters for formulas like these, we walk downhill instead — that is **gradient descent**, the topic of Module 2.

### 4.4 · The extrapolation trap — slide 30

In [ ]:
print(f"20 hours of study → {predict(20, m3, b3):.1f} marks out of 100")

**Think.** The maths is correct, so what went wrong? Write two sentences.

*Your answer:* …

## Part 5 · Real data — the seven steps and a pipeline — slides 15, 41–42

scikit-learn ships a small medical dataset: 442 diabetes patients, ten baseline measurements each, and a **target** that measures how far the disease progressed one year later. It is bundled with the library, so no download is needed.

> Note: the ten features are already centred and scaled by the dataset's authors, so their values look small and some are negative. That is expected.

### Step 1 · Gathering data

In [ ]:
from sklearn.datasets import load_diabetes

df = load_diabetes(as_frame=True).frame
print(df.shape)
df.head()

### Step 2 · Preparing (and exploring) the data
Which single feature moves most with the target? Correlation gives a first hint.

In [ ]:
print(df.isna().sum().sum(), "missing values")
print(df.corr()["target"].drop("target").sort_values(ascending=False).round(2))

plt.scatter(df["bmi"], df["target"], s=10, color="black", alpha=0.5)
plt.xlabel("bmi  (scaled)"); plt.ylabel("disease progression"); plt.title("One feature, lots of noise")
plt.show()

### Steps 3–5 · TODO — simple linear regression on `bmi`
1. Make `X` from the `bmi` column **as a 2-D DataFrame** (use double brackets) and `y` from `target`.
2. Split 80 / 20 with `random_state=42` — **before** fitting anything.
3. Fit `LinearRegression` on the training set only, predict the test set, and report test RMSE and R².

In [ ]:
from sklearn.model_selection import train_test_split

# TODO
X = ...                          # the bmi column, 2-D
y = ...                          # the target column
X_train, X_test, y_train, y_test = ...     # 80/20 split, random_state=42

slr = ...                        # LinearRegression fitted on the TRAINING data
pred = ...                       # predictions for X_test

slr_rmse, slr_r2 = ..., ...      # root_mean_squared_error and r2_score on the TEST data
print("slope", slr.coef_, "intercept", slr.intercept_, "| test RMSE", slr_rmse, "test R²", slr_r2)

In [ ]:
# Check
assert X.shape == (442, 1), "X should be 442 rows × 1 column — did you use df[['bmi']]?"
assert len(X_test) == 89, "Use test_size=0.2"
assert 0.15 < slr_r2 < 0.35, "Test R² should be about 0.23 — did you fit on the training set and score on the test set?"
print(f"PASS: one feature explains about {slr_r2:.0%} of the variation on unseen patients")

Compare that with Example 3's R² of 0.92. Real, noisy data with one feature rarely scores high — and an honest 0.23 on a test set is worth more than a flattering number on training data.

### Steps 3–6 at scale · all ten features in a pipeline
Now the pattern from slide 41: preprocessing and model inside one `Pipeline`, cross-validated tuning with `GridSearchCV`, and the test set touched **once** at the end. `Ridge` is linear regression plus a penalty on large coefficients; `alpha` sets its strength and is our hyperparameter.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

X_all = df.drop(columns="target")
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_all, df["target"], test_size=0.2, random_state=42)

pipe = Pipeline([("scale", StandardScaler()), ("model", Ridge())])
grid = GridSearchCV(pipe, {"model__alpha": [0.01, 0.1, 1, 10, 100]}, cv=5, scoring="r2")
grid.fit(Xa_train, ya_train)                                   # steps 4 + 6: train and tune

pred_all = grid.predict(Xa_test)                               # step 5: evaluate once
mlr_rmse, mlr_r2 = root_mean_squared_error(ya_test, pred_all), r2_score(ya_test, pred_all)
print("best alpha:", grid.best_params_["model__alpha"], "| mean cross-validated R²:", round(grid.best_score_, 3))

summary = pd.DataFrame({"test RMSE": [slr_rmse, mlr_rmse], "test R²": [slr_r2, mlr_r2]},
                       index=["SLR · bmi only", "Ridge pipeline · 10 features"]).round(3)
summary

Which features does the model lean on? Because every feature was scaled, the coefficient sizes are comparable.

In [ ]:
coefs = pd.Series(grid.best_estimator_.named_steps["model"].coef_, index=X_all.columns).sort_values()
coefs.plot.barh(color="black")
plt.axvline(0, color="black", linewidth=0.8); plt.xlabel("coefficient (per 1 standard deviation)")
plt.title("What the Ridge model learned"); plt.show()

### Step 7 · Prediction — save the model, load it back, use it

In [ ]:
import joblib

joblib.dump(grid.best_estimator_, "diabetes_ridge.joblib")
loaded = joblib.load("diabetes_ridge.joblib")

new_patient = Xa_test.iloc[[0]]              # stands in for a new patient's measurements
print("prediction:", loaded.predict(new_patient).round(1), "| actual:", ya_test.iloc[0])

Because scaling lives **inside** the pipeline, the saved file takes raw measurements and applies exactly the preprocessing it was trained with. That is the golden rule from slide 41.

---
## Part 6 · Assessment tasks — slide 43

Submit this notebook with Parts 1–5 completed **and** the three tasks below.

### Task 1 · Fit a line on paper
Data: $x = 1, 2, 3, 4, 5$ and $y = 3, 5, 4, 7, 9$.
On paper (or in a markdown table), build the deviations table and find $\bar{x}, \bar{y}, m, b$; predict $y$ at $x = 6$; compute MAE, RMSE and $R^2$.
Then type **your hand answers** (2 decimal places) below and run the check. Do not use code to get them.

In [ ]:
# Task 1 — type the answers you worked out BY HAND
task1 = dict(m=..., b=..., y_at_6=..., MAE=..., RMSE=..., R2=...)

In [ ]:
# Check Task 1
tx, ty = np.array([1, 2, 3, 4, 5.]), np.array([3, 5, 4, 7, 9.])
tm, tb = fit_line(tx, ty)
te = ty - predict(tx, tm, tb)
truth = dict(m=tm, b=tb, y_at_6=predict(6, tm, tb), MAE=np.abs(te).mean(),
             RMSE=np.sqrt((te ** 2).mean()), R2=1 - (te ** 2).sum() / ((ty - ty.mean()) ** 2).sum())
for k, v in truth.items():
    yours = task1[k]
    if yours is ...:
        print(f"EMPTY {k:7s} not filled in yet")
    else:
        print(f"{'PASS' if abs(yours - v) <= 0.011 else 'FAIL'} {k:7s} yours = {yours}")

### Task 2 · Code it and plot it
Fit `LinearRegression` on Task 1's data. Print `coef_` and `intercept_` and confirm they match your hand answer. Then draw **one** chart with the points, the fitted line, and a dashed vertical line for each residual (reuse Part 4.1).

In [ ]:
# Task 2 — your code here

### Task 3 · Build it end to end
Using the diabetes dataset from Part 5, build **your own** pipeline — for example, choose three features you think matter, or try `LinearRegression` instead of `Ridge`. Split first, keep all preprocessing inside the `Pipeline`, report **test** RMSE and R², and compare with the results in Part 5.

In [ ]:
# Task 3 — your pipeline here

**Task 3 · Explain your numbers in three plain sentences** — as if to a doctor who has never studied ML. What does your RMSE mean in the target's own units? What does R² tell them? Would you trust this model on its own?

*Your answer:* …

---
### Before you submit
- [ ] Every Check cell passes
- [ ] Both *Think* questions in Parts 3 and 4 are answered
- [ ] Task 1 answers were worked out by hand
- [ ] Task 2 chart shows points, line and residuals
- [ ] Task 3 reports **test** RMSE and R² with three sentences of explanation
- [ ] *Runtime → Restart and run all* completes without errors

**Stretch (optional):** In Part 4, add one outlier — a student who studied 5 hours and scored 20 — refit, and watch how far one point drags the line. Which metric changes the most, MAE or RMSE? Why?

*Next · Module 2 — multiple and polynomial regression, and gradient descent.*